In [ ]:
import os

from pypdf import PdfReader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings


PDF_PATH = "가상Tech_업무가이드.pdf"
DB_PATH = "./chroma_db/basic"

# 1. PDF를 페이지별 Document로 변환
reader = PdfReader(PDF_PATH)

documents = [
    Document(
        page_content=page.extract_text() or "",
        metadata={
            "source": PDF_PATH,
            "page": page_index,
            "page_number": page_index + 1,
        },
    )
    for page_index, page in enumerate(reader.pages)
]

# 2. 문서 분할
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

docs = splitter.split_documents(documents)

# 3. 임베딩 모델
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

# 4. 저장된 ChromaDB가 있으면 재사용
if os.path.exists(f"{DB_PATH}/chroma.sqlite3"):
    vectorstore = Chroma(
        collection_name="virtualtech-guide",
        persist_directory=DB_PATH,
        embedding_function=embeddings,
    )
    print("기존 ChromaDB를 불러왔습니다.")

else:
    vectorstore = Chroma.from_documents(
        documents=docs,
        embedding=embeddings,
        collection_name="virtualtech-guide",
        persist_directory=DB_PATH,
    )
    print("새 ChromaDB를 생성했습니다.")

# 5. 검색기 생성
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 4}
)

retriever